# Model Optimization and Comparison

This notebook optimizes the XGBoost model with tuned hyperparameters and compares it against the saved baseline model. If the optimized model performs better, you can choose to save it.

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv('data/loan_applications.csv')
df2 = df[df['actual_outcome']!='ongoing'].copy()

# Preprocess
df_clean = df2.copy()
df_clean['documented_monthly_income'] = df_clean['documented_monthly_income'].fillna(df_clean['stated_monthly_income'])
df_clean['target'] = (df_clean['actual_outcome']=='defaulted').astype(int)
df_clean[['bank_has_overdrafts','bank_has_consistent_deposits']] = df_clean[['bank_has_overdrafts','bank_has_consistent_deposits']].astype(int)
df_clean = pd.get_dummies(df_clean, columns=['employment_status'], drop_first=True)
drop_cols = ['applicant_id','actual_outcome','days_to_default','rule_based_score','rule_based_decision']
df_clean = df_clean.drop(columns=drop_cols)

# Train/test split
X = df_clean.drop('target', axis=1)
y = df_clean['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Class weight
cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
spw = cw[1] / cw[0]

print(f'Data shape: {X.shape}')
print(f'Target distribution: {y.value_counts().to_dict()}')
print(f'Class weight (scale_pos_weight): {spw:.3f}')

Data shape: (1836, 11)
Target distribution: {0: 1291, 1: 545}
Class weight (scale_pos_weight): 2.367


In [2]:
# Load baseline model
try:
    baseline_model = joblib.load('trained_model.pkl')
    baseline_pred = baseline_model.predict(X_test)
    baseline_proba = baseline_model.predict_proba(X_test)[:, 1]
    print('Loaded baseline model from trained_model.pkl')
except FileNotFoundError:
    print('Model file not found. Please run main training first.')
    raise

# Baseline metrics
baseline_metrics = {
    'Precision': precision_score(y_test, baseline_pred),
    'Recall': recall_score(y_test, baseline_pred),
    'F1': f1_score(y_test, baseline_pred),
    'AUC-ROC': roc_auc_score(y_test, baseline_proba),
    'Test Accuracy': baseline_model.score(X_test, y_test)
}

print('\nBASELINE MODEL METRICS:')
for metric, value in baseline_metrics.items():
    print(f'{metric}: {value:.4f}')

Loaded baseline model from trained_model.pkl

BASELINE MODEL METRICS:
Precision: 0.4839
Recall: 0.5505
F1: 0.5150
AUC-ROC: 0.6931
Test Accuracy: 0.6929


In [3]:
# Train optimized model with better hyperparameters
print('Training optimized model...')

# Tuned hyperparameters based on class imbalance and default prediction
optimized_model = xgb.XGBClassifier(
    n_estimators=200,           # More trees
    max_depth=5,                # Shallower trees (reduce overfitting)
    learning_rate=0.05,         # Lower learning rate (more robust)
    subsample=0.8,              # Use 80% of samples per tree (regularization)
    colsample_bytree=0.8,       # Use 80% of features per tree (regularization)
    scale_pos_weight=spw,       # Handle class imbalance
    min_child_weight=5,         # Prevent overfitting on small groups
    reg_alpha=0.1,              # L1 regularization
    reg_lambda=1.0,             # L2 regularization
    random_state=42,
    verbosity=0,
    eval_metric='logloss'
)

optimized_model.fit(X_train, y_train)
print('Optimized model training complete')

# Optimized predictions
optimized_pred = optimized_model.predict(X_test)
optimized_proba = optimized_model.predict_proba(X_test)[:, 1]

# Optimized metrics
optimized_metrics = {
    'Precision': precision_score(y_test, optimized_pred),
    'Recall': recall_score(y_test, optimized_pred),
    'F1': f1_score(y_test, optimized_pred),
    'AUC-ROC': roc_auc_score(y_test, optimized_proba),
    'Test Accuracy': optimized_model.score(X_test, y_test)
}

print('\nOPTIMIZED MODEL METRICS:')
for metric, value in optimized_metrics.items():
    print(f'{metric}: {value:.4f}')

Training optimized model...
Optimized model training complete

OPTIMIZED MODEL METRICS:
Precision: 0.4853
Recall: 0.6055
F1: 0.5388
AUC-ROC: 0.7174
Test Accuracy: 0.6929


In [4]:
# Comparison
import pandas as pd

comparison_df = pd.DataFrame({
    'Metric': list(baseline_metrics.keys()),
    'Baseline': list(baseline_metrics.values()),
    'Optimized': list(optimized_metrics.values())
})

comparison_df['Improvement'] = comparison_df['Optimized'] - comparison_df['Baseline']
comparison_df['% Change'] = (comparison_df['Improvement'] / comparison_df['Baseline'] * 100).round(2)

print('\n' + '='*70)
print('MODEL COMPARISON: BASELINE vs OPTIMIZED')
print('='*70)
print(comparison_df.to_string(index=False))

# Summary
better_metrics = (comparison_df['Improvement'] > 0).sum()
total_metrics = len(comparison_df)
print(f'\nOptimized model better on {better_metrics}/{total_metrics} metrics')


MODEL COMPARISON: BASELINE vs OPTIMIZED
       Metric  Baseline  Optimized  Improvement  % Change
    Precision  0.483871   0.485294     0.001423      0.29
       Recall  0.550459   0.605505     0.055046     10.00
           F1  0.515021   0.538776     0.023754      4.61
      AUC-ROC  0.693139   0.717403     0.024264      3.50
Test Accuracy  0.692935   0.692935     0.000000      0.00

Optimized model better on 4/5 metrics


In [5]:
# Confusion matrices comparison
baseline_cm = confusion_matrix(y_test, baseline_pred)
optimized_cm = confusion_matrix(y_test, optimized_pred)

print('\nCONFUSION MATRICES:')
print('\nBaseline:')
print(f'TN={baseline_cm[0,0]}, FP={baseline_cm[0,1]}')
print(f'FN={baseline_cm[1,0]}, TP={baseline_cm[1,1]}')

print('\nOptimized:')
print(f'TN={optimized_cm[0,0]}, FP={optimized_cm[0,1]}')
print(f'FN={optimized_cm[1,0]}, TP={optimized_cm[1,1]}')

# FPR/FNR
baseline_fpr = baseline_cm[0,1] / (baseline_cm[0,1] + baseline_cm[0,0])
baseline_fnr = baseline_cm[1,0] / (baseline_cm[1,0] + baseline_cm[1,1])

optimized_fpr = optimized_cm[0,1] / (optimized_cm[0,1] + optimized_cm[0,0])
optimized_fnr = optimized_cm[1,0] / (optimized_cm[1,0] + optimized_cm[1,1])

print(f'\nBaseline FPR: {baseline_fpr:.4f}, FNR: {baseline_fnr:.4f}')
print(f'Optimized FPR: {optimized_fpr:.4f}, FNR: {optimized_fnr:.4f}')
print(f'\nFPR Change: {optimized_fpr - baseline_fpr:+.4f}')
print(f'FNR Change: {optimized_fnr - baseline_fnr:+.4f}')


CONFUSION MATRICES:

Baseline:
TN=195, FP=64
FN=49, TP=60

Optimized:
TN=189, FP=70
FN=43, TP=66

Baseline FPR: 0.2471, FNR: 0.4495
Optimized FPR: 0.2703, FNR: 0.3945

FPR Change: +0.0232
FNR Change: -0.0550


In [6]:
# Cross-validation comparison
from sklearn.model_selection import cross_validate

print('Running cross-validation (5-fold)...')

baseline_cv = cross_validate(baseline_model, X_train, y_train, cv=5, 
                             scoring=['precision', 'recall', 'f1', 'roc_auc'])
optimized_cv = cross_validate(optimized_model, X_train, y_train, cv=5,
                              scoring=['precision', 'recall', 'f1', 'roc_auc'])

cv_comparison = pd.DataFrame({
    'Metric': ['Precision', 'Recall', 'F1', 'AUC-ROC'],
    'Baseline Mean': [
        baseline_cv['test_precision'].mean(),
        baseline_cv['test_recall'].mean(),
        baseline_cv['test_f1'].mean(),
        baseline_cv['test_roc_auc'].mean()
    ],
    'Baseline Std': [
        baseline_cv['test_precision'].std(),
        baseline_cv['test_recall'].std(),
        baseline_cv['test_f1'].std(),
        baseline_cv['test_roc_auc'].std()
    ],
    'Optimized Mean': [
        optimized_cv['test_precision'].mean(),
        optimized_cv['test_recall'].mean(),
        optimized_cv['test_f1'].mean(),
        optimized_cv['test_roc_auc'].mean()
    ],
    'Optimized Std': [
        optimized_cv['test_precision'].std(),
        optimized_cv['test_recall'].std(),
        optimized_cv['test_f1'].std(),
        optimized_cv['test_roc_auc'].std()
    ]
})

print('\nCROSS-VALIDATION RESULTS (5-fold):')
print(cv_comparison.round(4).to_string(index=False))

Running cross-validation (5-fold)...

CROSS-VALIDATION RESULTS (5-fold):
   Metric  Baseline Mean  Baseline Std  Optimized Mean  Optimized Std
Precision         0.4496        0.0235          0.4549         0.0271
   Recall         0.4404        0.0396          0.5183         0.0352
       F1         0.4433        0.0198          0.4828         0.0121
  AUC-ROC         0.6622        0.0121          0.6881         0.0144


In [7]:
# Feature importance comparison
from sklearn.inspection import permutation_importance

print('Computing permutation importances for both models...')

baseline_perm = permutation_importance(baseline_model, X_test, y_test, n_repeats=5, random_state=42)
optimized_perm = permutation_importance(optimized_model, X_test, y_test, n_repeats=5, random_state=42)

importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Baseline Importance': baseline_perm.importances_mean,
    'Optimized Importance': optimized_perm.importances_mean
}).sort_values('Optimized Importance', ascending=False)

print('\nTOP 10 FEATURES (by Optimized Model):')
print(importance_df.head(10).round(6).to_string(index=False))

Computing permutation importances for both models...

TOP 10 FEATURES (by Optimized Model):
                        Feature  Baseline Importance  Optimized Importance
        num_documents_submitted             0.056522              0.048913
                    loan_amount             0.023370              0.038587
          stated_monthly_income            -0.004348              0.027174
            monthly_withdrawals             0.033696              0.023913
            bank_has_overdrafts             0.019565              0.022283
               monthly_deposits             0.052174              0.020652
      documented_monthly_income             0.011413              0.015217
            bank_ending_balance             0.010870              0.005435
   bank_has_consistent_deposits            -0.000543              0.000000
employment_status_self_employed            -0.001630             -0.006522


In [8]:
# Decision: save optimized model if better
print('\n' + '='*70)
print('RECOMMENDATION')
print('='*70)

# Calculate weighted improvement
auc_improvement = optimized_metrics['AUC-ROC'] - baseline_metrics['AUC-ROC']
f1_improvement = optimized_metrics['F1'] - baseline_metrics['F1']
recall_improvement = optimized_metrics['Recall'] - baseline_metrics['Recall']

print(f'\nAUC-ROC improvement: {auc_improvement:+.4f}')
print(f'F1-Score improvement: {f1_improvement:+.4f}')
print(f'Recall improvement: {recall_improvement:+.4f}')

if auc_improvement > 0.01 or (auc_improvement > 0 and f1_improvement > 0):
    print('\n✓ OPTIMIZED MODEL IS BETTER')
    print('\nReasons:')
    if auc_improvement > 0:
        print(f'  - Better discrimination (AUC-ROC +{auc_improvement:.4f})')
    if recall_improvement > 0:
        print(f'  - Better at catching defaults (Recall +{recall_improvement:.4f})')
    if f1_improvement > 0:
        print(f'  - Better balanced performance (F1 +{f1_improvement:.4f})')
else:
    print('\n✗ BASELINE MODEL IS COMPARABLE OR BETTER')
    print('Consider keeping the baseline model')


RECOMMENDATION

AUC-ROC improvement: +0.0243
F1-Score improvement: +0.0238
Recall improvement: +0.0550

✓ OPTIMIZED MODEL IS BETTER

Reasons:
  - Better discrimination (AUC-ROC +0.0243)
  - Better at catching defaults (Recall +0.0550)
  - Better balanced performance (F1 +0.0238)


In [ ]:
# SAVE OPTIMIZED MODEL (uncomment if you want to overwrite)
# joblib.dump(optimized_model, 'trained_model.pkl')
# print('\nOptimized model saved to trained_model.pkl')

print('\nTo save the optimized model, uncomment the joblib.dump line above and re-run this cell.')
print('Or run the cell below to save it.')

In [9]:
# Run this cell to save the optimized model
save_optimized = True  # Set to True to save

if save_optimized:
    joblib.dump(optimized_model, 'trained_model.pkl')
    print('✓ Optimized model saved to trained_model.pkl')
    print('\nPresentation notebook will now use the optimized model.')
else:
    print('Optimized model NOT saved. Baseline remains in use.')

✓ Optimized model saved to trained_model.pkl

Presentation notebook will now use the optimized model.
